imports 

In [1]:
import sys
sys.path.append('../src')

import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from retriever    import load_retriever, retrieve, format_context
from generator    import load_generator, build_prompt, generate_answer, PROMPT_TEMPLATE
from rag_pipeline import RAGPipeline

inspect prompt template

In [2]:
print(PROMPT_TEMPLATE)

You are a financial analyst assistant for CrediTrust Financial.
Your task is to answer questions about customer complaints.

Use ONLY the complaint excerpts provided below to formulate your answer.
Do not use any outside knowledge. If the context does not contain enough
information to answer the question, say: "I don't have enough information
in the retrieved complaints to answer this question confidently."

Where possible, reference which product or issue category the evidence comes from.

Context:
{context}

Question: {question}

Answer:


load pipeline

In [5]:
rag = RAGPipeline(
    vector_store_dir='../vector_store',
    generator_model='google/flan-t5-large',
    k=5,
    max_new_tokens=300,
    device=-1,   
)

Initialising RAG pipeline...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3433.58it/s]


Loaded embedding model: sentence-transformers/all-MiniLM-L6-v2
Embedding dimensions: 384
Retriever ready — 35,831 vectors loaded
Loading tokenizer and model: google/flan-t5-large ...


KeyboardInterrupt: 

retriver smoke test

In [ ]:
test_query = "problems with credit card billing charges"

results = retrieve(
    query=test_query,
    index=rag.index,
    chunks_df=rag.chunks_df,
    model=rag.embed_model,
    k=5
)

for r in results:
    print(f"Rank {r['rank']} | Score: {r['score']:.4f} | "
          f"Product: {r['product_category']} | Issue: {r['issue']}")
    print(f"  {r['chunk_text'][:200]}\n")

NameError: name 'rag' is not defined

test full rag pipeline

In [ ]:
result = rag.ask("Why are customers complaining about credit cards?")

print("=" * 60)
print("QUESTION:", result['question'])
print("=" * 60)
print("\nANSWER:\n", result['answer'])
print("\nSOURCES:")
for s in result['sources']:
    print(f"  [{s['rank']}] {s['product_category']} | {s['issue']} "
          f"| Score: {s['score']:.4f}")

product-filtered query

In [ ]:
result_filtered = rag.ask(
    "What are the main issues with money transfers?",
    product_filter="Money Transfer"
)

print("ANSWER:\n", result_filtered['answer'])
print("\nSOURCES:")
for s in result_filtered['sources']:
    print(f"  [{s['rank']}] {s['product_category']} | {s['issue']}")

Qualitative Evaluation
evaluation questions

In [ ]:
eval_questions = [
    # Credit Card
    {"question": "Why are people unhappy with their credit cards?",
     "product_filter": None},
    {"question": "What billing issues do credit card customers report?",
     "product_filter": "Credit Card"},

    # Personal Loan
    {"question": "What problems do customers have with personal loans?",
     "product_filter": None},
    {"question": "Are there complaints about loan interest rates being wrong?",
     "product_filter": "Personal Loan"},

    # Savings Account
    {"question": "What are the most common savings account complaints?",
     "product_filter": "Savings Account"},
    {"question": "Do customers complain about unauthorised account access?",
     "product_filter": None},

    # Money Transfer
    {"question": "Why do money transfers fail or get delayed?",
     "product_filter": "Money Transfer"},

    # Cross-product
    {"question": "Which product has the most fraud-related complaints?",
     "product_filter": None},
]

run evaluation

In [ ]:
eval_results = []

for item in eval_questions:
    result = rag.ask(
        question=item['question'],
        product_filter=item.get('product_filter')
    )

    # Pick top 2 sources for the table
    top_sources = result['sources'][:2]
    source_str  = " | ".join([
        f"{s['product_category']} — {s['issue'][:40]}"
        for s in top_sources
    ])

    eval_results.append({
        'Question':           result['question'],
        'Product Filter':     item.get('product_filter') or 'None',
        'Generated Answer':   result['answer'],
        'Top Sources':        source_str,
        'Quality Score (1-5)': '',   
        'Comments':           '',    
    })

    print(f"Q: {result['question']}")
    print(f"A: {result['answer'][:200]}\n")

eval_df = pd.DataFrame(eval_results)
eval_df

export evaluation table

In [ ]:
eval_df.to_csv('../data/processed/rag_evaluation.csv', index=False)
print("Evaluation table saved to data/processed/rag_evaluation.csv")